In [3]:
import cv2
import numpy as np
from google.colab import files
from IPython.display import display
import matplotlib.pyplot as plt
from PIL import Image
import io

In [16]:
def gstreamer_pipeline():
    return (
        "nvarguscamerasrc ! "
        "video/x-raw(memory:NVMM), width=1280, height=720, format=NV12, framerate=30/1 ! "
        "nvvidconv ! video/x-raw, format=BGRx ! "
        "videoconvert ! video/x-raw, format=BGR ! appsink"
    )

cap = cv2.VideoCapture(gstreamer_pipeline(), cv2.CAP_GSTREAMER)

# Initialize frame and mask to avoid UnboundLocalError if loop never runs
frame = None
mask = None

while True:
    ret, current_frame = cap.read() # Renamed to avoid confusion with outer 'frame' for display
    if not ret:
        print("Failed to grab frame from camera or end of stream.")
        break

    frame = current_frame # Keep the last valid frame

    # BGR → HSV 변환 (색상 감지에 유리)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # 감지할 색상 범위 설정 (초록색)
    lower = np.array([40, 50, 50])
    upper = np.array([80, 255, 255])

    # 마스크 생성
    mask = cv2.inRange(hsv, lower, upper)

    # 노이즈 제거
    mask = cv2.erode(mask, None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)
    # 윤곽선 찾기
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 500:  # 너무 작은 건 무시
            continue

        # 바운딩 박스 그리기
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame, f"Object ({area:.0f}px)", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Removed cv2.imshow and cv2.waitKey
    # if cv2.waitKey(1) & 0xFF == ord('q'):
    #     break

# Display the last processed frame and mask using matplotlib after the loop
if frame is not None and mask is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title("감지 결과 (마지막 프레임)")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("마스크 (마지막 프레임)")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No frames were processed.")


cap.release()
# cv2.destroyAllWindows() is removed

Failed to grab frame from camera or end of stream.
No frames were processed.
